# Montowanie Google Drive


KROK 1 — Montowanie Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('✅ Google Drive zamontowany')

# Instalacja zależności i pobieranie wag modelu

KROK 2 — Instalacja zależności i pobieranie wag modeli

In [ ]:
import os
import subprocess

# Instalacja bibliotek
os.system('pip install -q basicsr facexlib gfpgan realesrgan')

# Klonowanie Real-ESRGAN (jeśli nie istnieje)
if not os.path.exists('/content/Real-ESRGAN'):
    os.system('git clone https://github.com/xinntao/Real-ESRGAN.git /content/Real-ESRGAN')
    os.chdir('/content/Real-ESRGAN')
    os.system('pip install -q -r requirements.txt')
    os.system('python setup.py develop --quiet')
    os.chdir('/content')

print('✅ Zależności zainstalowane')

# Pobieranie wag modeli
os.makedirs('/content/weights', exist_ok=True)

if not os.path.exists('/content/weights/RealESRGAN_x4plus.pth'):
    os.system('wget -q -O /content/weights/RealESRGAN_x4plus.pth '
              'https://github.com/xinntao/Real-ESRGAN/releases/download/v0.1.0/RealESRGAN_x4plus.pth')
    print('✅ Real-ESRGAN wagi pobrane')
else:
    print('ℹ️  Real-ESRGAN wagi już istnieją')

if not os.path.exists('/content/weights/RRDB_ESRGAN_x4.pth'):
    os.system('wget -q -O /content/weights/RRDB_ESRGAN_x4.pth '
              'https://github.com/xinntao/ESRGAN/releases/download/v0.1.1/RRDB_ESRGAN_x4.pth')
    print('✅ ESRGAN classic wagi pobrane')
else:
    print('ℹ️  ESRGAN classic wagi już istnieją')

if not os.path.exists('/content/weights/EDSR_x4.pth'):
    os.system('wget -q -O /content/weights/EDSR_x4.pth '
              'https://github.com/xinntao/BasicSR/releases/download/v1.4.2/'
              'EDSR_Lx4_f256b32_DIV2K_official-76ee1c8f.pth')
    print('✅ EDSR wagi pobrane')
else:
    print('ℹ️  EDSR wagi już istnieją')

print('\n✅ Wszystkie wagi modeli gotowe')

# Konfiguracja ścieżki

KROK 3 — KONFIGURACJA  ← zmień ścieżki tutaj

In [ ]:
INPUT_DIR    = '/content/drive/MyDrive/zdjecia_wejsciowe'  # <- folder źródłowy
OUTPUT_DIR   = '/content/drive/MyDrive/zdjecia_wynikowe'   # <- folder docelowy
CROP_SIZE    = 256   # rozmiar wycinanego fragmentu (px)
SCALE_FACTOR = 4     # skala: 256 → 64 → 256

os.makedirs(OUTPUT_DIR, exist_ok=True)

extensions = ('.jpg', '.jpeg', '.png', '.bmp', '.tiff', '.webp')
if not os.path.exists(INPUT_DIR):
    print(f'⚠️  Folder wejściowy nie istnieje: {INPUT_DIR}')
else:
    images = [f for f in sorted(os.listdir(INPUT_DIR)) if f.lower().endswith(extensions)]
    print(f'✅ Folder wejściowy : {INPUT_DIR}')
    print(f'✅ Folder wyjściowy : {OUTPUT_DIR}')
    print(f'📷 Znalezione zdjęcia: {len(images)}')
    for img in images:
        print(f'   • {img}')

# Inne importy

KROK 4 — Importy i sprawdzenie GPU

In [ ]:
import sys
import json
import time
import numpy as np
from PIL import Image, ImageFilter
import cv2
import torch
from pathlib import Path

print(f'PyTorch: {torch.__version__}')
print(f'CUDA dostępna: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Urządzenie: {DEVICE}')

# Inicjalizacja modeli

KROK 5 — Inicjalizacja modeli (ładowane raz)

In [ ]:
from realesrgan import RealESRGANer
from basicsr.archs.rrdbnet_arch import RRDBNet
from basicsr.archs.edsr_arch import EDSR

# --- Real-ESRGAN ---
realesrgan_model = RRDBNet(
    num_in_ch=3, num_out_ch=3, num_feat=64,
    num_block=23, num_grow_ch=32, scale=SCALE_FACTOR
)
realesrgan_upsampler = RealESRGANer(
    scale=SCALE_FACTOR,
    model_path='/content/weights/RealESRGAN_x4plus.pth',
    model=realesrgan_model,
    tile=0, tile_pad=10, pre_pad=0,
    half=(DEVICE == 'cuda')
)
print('✅ Real-ESRGAN załadowany')

# --- ESRGAN classic ---
esrgan_classic_model = RRDBNet(
    num_in_ch=3, num_out_ch=3, num_feat=64,
    num_block=23, num_grow_ch=32, scale=SCALE_FACTOR
)
esrgan_classic_upsampler = RealESRGANer(
    scale=SCALE_FACTOR,
    model_path='/content/weights/RRDB_ESRGAN_x4.pth',
    model=esrgan_classic_model,
    tile=0, tile_pad=10, pre_pad=0,
    half=(DEVICE == 'cuda')
)
print('✅ ESRGAN classic załadowany')

# --- EDSR ---
edsr_model = EDSR(
    num_in_ch=3, num_out_ch=3,
    num_feat=256, num_block=32,
    upscale=SCALE_FACTOR, res_scale=0.1,
    img_range=255.0,
    rgb_mean=(0.4488, 0.4371, 0.4040)
)
edsr_checkpoint = torch.load('/content/weights/EDSR_x4.pth', map_location=torch.device(DEVICE))
state_dict = edsr_checkpoint.get('params', edsr_checkpoint.get('params_ema', edsr_checkpoint))
edsr_model.load_state_dict(state_dict, strict=True)
edsr_model.eval().to(DEVICE)
print('✅ EDSR załadowany')

# Definiowanie algorytmów

KROK 6 — Definicje funkcji algorytmów

In [ ]:
def crop_center(img: Image.Image, size: int = 256) -> Image.Image:
    """Wycina kwadrat size×size ze środka obrazu."""
    w, h = img.size
    if w < size or h < size:
        scale = size / min(w, h)
        new_w, new_h = int(w * scale) + 1, int(h * scale) + 1
        img = img.resize((new_w, new_h), Image.LANCZOS)
        w, h = img.size
    left = (w - size) // 2
    top  = (h - size) // 2
    return img.crop((left, top, left + size, top + size))


# ---- Downscaling ----

def downscale_nearest(img: Image.Image, factor: int) -> Image.Image:
    w, h = img.size
    return img.resize((w // factor, h // factor), Image.NEAREST)

def downscale_gaussian(img: Image.Image, factor: int, sigma: float = 1.5) -> Image.Image:
    blurred = img.filter(ImageFilter.GaussianBlur(radius=sigma))
    w, h = img.size
    return blurred.resize((w // factor, h // factor), Image.LANCZOS)

def downscale_bicubic(img: Image.Image, factor: int) -> Image.Image:
    w, h = img.size
    return img.resize((w // factor, h // factor), Image.BICUBIC)

def downscale_lanczos(img: Image.Image, factor: int) -> Image.Image:
    w, h = img.size
    return img.resize((w // factor, h // factor), Image.LANCZOS)


# ---- Upscaling ----

def upscale_nearest(img: Image.Image, factor: int) -> Image.Image:
    w, h = img.size
    return img.resize((w * factor, h * factor), Image.NEAREST)

def upscale_edsr(img: Image.Image) -> Image.Image:
    img_np = np.array(img).astype(np.float32) / 255.0
    tensor = torch.from_numpy(img_np).permute(2, 0, 1).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        output = edsr_model(tensor)
    output_np = output.squeeze(0).permute(1, 2, 0).clamp(0, 1).cpu().numpy()
    return Image.fromarray((output_np * 255).astype(np.uint8))

def upscale_realesrgan(img: Image.Image) -> Image.Image:
    img_bgr = cv2.cvtColor(np.array(img), cv2.COLOR_RGB2BGR)
    output_bgr, _ = realesrgan_upsampler.enhance(img_bgr, outscale=SCALE_FACTOR)
    return Image.fromarray(cv2.cvtColor(output_bgr, cv2.COLOR_BGR2RGB))

def upscale_esrgan_classic(img: Image.Image) -> Image.Image:
    img_bgr = cv2.cvtColor(np.array(img), cv2.COLOR_RGB2BGR)
    output_bgr, _ = esrgan_classic_upsampler.enhance(img_bgr, outscale=SCALE_FACTOR)
    return Image.fromarray(cv2.cvtColor(output_bgr, cv2.COLOR_BGR2RGB))


# ---- 4 pary algorytmów ----

ALGORITHM_PAIRS = [
    {
        'pair_id':   1,
        'down_name': 'Nearest Neighbor',
        'down_abbr': 'NN',
        'up_name':   'Nearest Neighbor',
        'up_abbr':   'NN',
        'downscale': downscale_nearest,
        'upscale':   upscale_nearest,
    },
    {
        'pair_id':   2,
        'down_name': 'Gaussian (sigma=1.5)',
        'down_abbr': 'GAUSS',
        'up_name':   'EDSR',
        'up_abbr':   'EDSR',
        'downscale': downscale_gaussian,
        'upscale':   upscale_edsr,
    },
    {
        'pair_id':   3,
        'down_name': 'Bicubic',
        'down_abbr': 'BIC',
        'up_name':   'Real-ESRGAN',
        'up_abbr':   'RESRGAN',
        'downscale': downscale_bicubic,
        'upscale':   upscale_realesrgan,
    },
    {
        'pair_id':   4,
        'down_name': 'Lanczos',
        'down_abbr': 'LANCZOS',
        'up_name':   'ESRGAN (classic)',
        'up_abbr':   'ESRGAN',
        'downscale': downscale_lanczos,
        'upscale':   upscale_esrgan_classic,
    },
]

print('✅ Pary algorytmów:')
for p in ALGORITHM_PAIRS:
    print(f"  Para {p['pair_id']}: {p['down_name']} → {p['up_name']}")

# Pipeline

KROK 7 — Główny pipeline

In [ ]:
image_files = sorted([
    f for f in os.listdir(INPUT_DIR)
    if f.lower().endswith(extensions)
])

if not image_files:
    print('⚠️  Brak zdjęć w folderze wejściowym!')
else:
    total = len(image_files)
    print(f'📷 Liczba zdjęć do przetworzenia: {total}')
    print(f'📁 Wyjście: {OUTPUT_DIR}\n')
    print('=' * 60)

    pipeline_start = time.time()

    for img_idx, filename in enumerate(image_files, start=1):
        img_path = os.path.join(INPUT_DIR, filename)
        print(f'\n[{img_idx}/{total}] Przetwarzam: {filename}')

        try:
            # Wczytaj i wytnij kadr 256×256
            raw  = Image.open(img_path).convert('RGB')
            crop = crop_center(raw, CROP_SIZE)

            # Zapisz oryginalne kadrowanie
            orig_name      = f'{img_idx}_orig'
            orig_img_path  = os.path.join(OUTPUT_DIR, f'{orig_name}.png')
            orig_json_path = os.path.join(OUTPUT_DIR, f'{orig_name}.json')

            crop.save(orig_img_path, 'PNG')

            orig_meta = {
                'numer_zdjecia':   img_idx,
                'plik_zrodlowy':   filename,
                'typ':             'oryginal',
                'opis':            'Oryginalne kadrowanie 256x256 px bez zmiany rozdzielczosci',
                'algorytm_down':   None,
                'algorytm_up':     None,
                'nazwa_pliku_img': f'{orig_name}.png',
                'rozmiar_px':      list(crop.size),
            }
            with open(orig_json_path, 'w', encoding='utf-8') as jf:
                json.dump(orig_meta, jf, ensure_ascii=False, indent=2)

            print(f'  ✓ Oryginał zapisany: {orig_name}.png')

            # Przetwórz przez każdą z 4 par
            for pair in ALGORITHM_PAIRS:
                pair_id = pair['pair_id']
                t0 = time.time()

                # Downscaling
                if pair_id == 2:
                    low_res = pair['downscale'](crop, SCALE_FACTOR, sigma=1.5)
                else:
                    low_res = pair['downscale'](crop, SCALE_FACTOR)

                # Upscaling
                if pair_id == 1:
                    result = pair['upscale'](low_res, SCALE_FACTOR)
                else:
                    result = pair['upscale'](low_res)

                elapsed = time.time() - t0

                # Zapis obrazu i JSON
                out_name      = f'{img_idx}_{pair_id}'
                out_img_path  = os.path.join(OUTPUT_DIR, f'{out_name}.png')
                out_json_path = os.path.join(OUTPUT_DIR, f'{out_name}.json')

                result.save(out_img_path, 'PNG')

                meta = {
                    'numer_zdjecia':        img_idx,
                    'plik_zrodlowy':        filename,
                    'typ':                  'przetworzone',
                    'numer_pary':           pair_id,
                    'algorytm_down': {
                        'nazwa': pair['down_name'],
                        'skrot': pair['down_abbr'],
                    },
                    'algorytm_up': {
                        'nazwa': pair['up_name'],
                        'skrot': pair['up_abbr'],
                    },
                    'skala':                SCALE_FACTOR,
                    'rozmiar_low_res_px':   list(low_res.size),
                    'rozmiar_wynikowy_px':  list(result.size),
                    'czas_przetwarzania_s': round(elapsed, 3),
                    'nazwa_pliku_img':      f'{out_name}.png',
                }
                with open(out_json_path, 'w', encoding='utf-8') as jf:
                    json.dump(meta, jf, ensure_ascii=False, indent=2)

                print(f'  ✓ Para {pair_id} ({pair["down_abbr"]}→{pair["up_abbr"]}): '
                      f'{out_name}.png  [{elapsed:.2f}s]')

        except Exception as exc:
            print(f'  ❌ Błąd przy przetwarzaniu {filename}: {exc}')
            import traceback; traceback.print_exc()

    total_time = time.time() - pipeline_start
    files_per_img = len(ALGORITHM_PAIRS) + 1
    print('\n' + '=' * 60)
    print('🏁 Pipeline zakończony!')
    print(f'   Przetworzonych zdjęć : {total}')
    print(f'   Obrazów PNG          : {total * files_per_img}')
    print(f'   Plików JSON          : {total * files_per_img}')
    print(f'   Czas całkowity       : {total_time:.1f}s ({total_time/60:.1f} min)')
    print(f'   Wyniki w             : {OUTPUT_DIR}')


# Wyniki

KROK 8 — Podgląd wyników

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

PREVIEW_IDX = 1  # <- zmień numer, żeby zobaczyć inne zdjęcie

labels = {
    'orig': 'Oryginał (256×256)',
    '1':    'Para 1: NN → NN',
    '2':    'Para 2: Gaussian → EDSR',
    '3':    'Para 3: Bicubic → Real-ESRGAN',
    '4':    'Para 4: Lanczos → ESRGAN',
}

fig, axes = plt.subplots(1, 5, figsize=(22, 5))
fig.suptitle(f'Podgląd wyników — zdjęcie #{PREVIEW_IDX}', fontsize=14)

for ax, suffix in zip(axes, ['orig', '1', '2', '3', '4']):
    path = os.path.join(OUTPUT_DIR, f'{PREVIEW_IDX}_{suffix}.png')
    if os.path.exists(path):
        ax.imshow(mpimg.imread(path))
    else:
        ax.text(0.5, 0.5, 'Brak pliku', ha='center', va='center')
    ax.set_title(labels[suffix], fontsize=9)
    ax.axis('off')

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, f'podglad_{PREVIEW_IDX}.png'), dpi=150, bbox_inches='tight')
plt.show()
print(f'Podgląd zapisany: podglad_{PREVIEW_IDX}.png')

# Podsumowanie plików w folderze wyjściowym
all_files  = sorted(os.listdir(OUTPUT_DIR))
png_files  = [f for f in all_files if f.endswith('.png')]
json_files = [f for f in all_files if f.endswith('.json')]

print(f'\n📁 Folder wyjściowy: {OUTPUT_DIR}')
print(f'🖼️  Obrazów PNG : {len(png_files)}')
print(f'📄 Plików JSON : {len(json_files)}')

# Przykładowy JSON (zdjęcie 1, para 2)
example_json = os.path.join(OUTPUT_DIR, '1_2.json')
if os.path.exists(example_json):
    print('\nPrzykładowy plik JSON (1_2.json):')
    with open(example_json) as jf:
        print(json.dumps(json.load(jf), indent=2, ensure_ascii=False))
